# HAR Stage 19B — Original-author BED source discovery and ±300 bp sequence-design gate

**No Google Drive. No large datasets. Run all cells.** This notebook embeds the 180-HAR source crosswalk and the 304 localized original sgRNA records from the verified Stage 19A–A3 analysis. It searches *public file metadata* for the Noble authors' original hg38 HAR BED, checks the published author code's 300-bp design extension, and exports all CSVs in a results ZIP.

**Critical scientific safeguard:** Catalog coordinates are provisional; 300/304 guides within ±300 bp does **not** establish original HAR interval identity. Four guides (HACNS336, HACNS643, HAR169×2) are further than ±300 bp from the provisional catalog core. The 28 HARs missing catalog ortholog pairs remain unresolved and are not included in this 304-guide audit.

**Optional**: If you obtain the exact author BED file, set `AUTHOR_BED_PATH` in the optional cell to compare coordinates. A matching filename does not verify provenance. If no BED found, the notebook completes and exports a clear unresolved status, rather than inventing confirmation.

Primary source: https://raw.githubusercontent.com/NoonanLab/Noble_et_al_2024/main/Library_Design/LibraryDesign_1_HAR_FlashFry.sh


In [ ]:

# Stage19B: Original-author sequence design verification, no Google Drive
from pathlib import Path
import base64, zlib, csv, io, json, re, requests, zipfile, time, hashlib
from collections import Counter, defaultdict
from urllib.parse import urljoin
from xml.etree import ElementTree as ET

ROOT=Path('/content') if Path('/content').exists() else (Path('/mnt/data') if Path('/mnt/data').exists() else Path.cwd())
OUT=ROOT/'HAR_Stage19B_Source_Discovery_Output'; OUT.mkdir(exist_ok=True)
HEADERS={'User-Agent':'Research-Reproducibility-HAR-Stage19B/1.0 (noncommercial academic study)','Accept':'application/json'}
PAYLOAD = {'S1_180HAR_Stage19B_DesignRule_Crosswalk.csv': 'eNrtXd1uXDeSvl9g36QnJlnFKtbeyY4zEeDYXsXJ7Fw1ZFmxtbAlQz8zybzaXuwj7StsFYukjMHouMHoLrQBn695pO42v0Oy/uv//ud/b86uz88v998fnexubk/fn0c52iu4vbvZnZ3enn68er+/eLf/fH31t4ubi6vL04+7D3efTi/3H95D2Z99uL761Ab0l65vQ3txfvku7M9/Pft4d3Pxt/Pd2YeLT5/3n08v31xfUfstH2u/5S/+6bcu91fXF+8v9DP37+8u3p3f7C8uv/wm+0+nnz9fXL7XH2z3/35x+0F/5vTu9sPV9f6zvs+ni8u7Gwhhf33+8fRW33V/e7Vv/7H737u6u71R8PXf+HT6q//O/uzq+nz/7kK//+XZ+f7t5/s3u7y63X+8Ojv9ePGP83f2lfu8Qp/hp3v9uYv3l/vru4/nfbY/X198Or3+rX/5819Pz273T59/u//l6u7y3a7/n9pPnV1dXb/Tqbk9V3rOL28vbn/b3VzdXet3ae+txF58vu2D7T/w7/+WflWqv4mRdq9PXv18/OPxq5dHL/bPjt4cvXj15/0PR2+efb9/+erN/vXJ8Q9HJ3/dP3v18rvjkx+ef7vT39sjFeXqOsadvkNkhAwNZcRxJ0rWkdSQAOyS/g27CKL/Pnv1w+ujN8dPXzzf/+X4zff7o5/efP/qZK9T/vT1/uQnHf7ym716+eKvu5evdvadfn5+cvzdsX6VD7e3n2/+48mT69O/f/NeOb97e3dzfn12dXmrU/HN2dWnJy+v9Am5fHH6VtFbneTz270+MCkkfPLp9OLyyYuLt9c2j9/Wyeov/dU+2nLYf/fx9ObDd9e/fXPzYXygf1j9hNPbD+eXp0rG1d0T/fE/vfnu6Y9/en199d/nZ7dPbnVRPTmVlOMvIML09u1ZzO9+ie/KL+HdL+fvTn8RLlSinCUpg5RUpkiJIeU697TjlBiVA7vmFPuokhNjqNeUQmODyyJjgwyY48Kee511nWXgkmMQcJA4jnGQop9YgcRMjY+w6NiiI+1+enny/MdXL37WPfHlq72E18/eDF6evdIJOPrz813749O506nZ/ysaXx8dn1QWn/989OKnozc6b3+UecxTzzUQtO29QOYYoF4h9U2/pIhFsl2zHS7rkf46FTRHReJUD2HW87QAFmXGQYl9WCBEO5kr0I3H2Uh6TCw+HuaDZ09f6eesnq4q+fi1pD6q2zwA1yuG1GWhzIuMLTLm2EhUZ30HJekfQAe5YBvmECjp1FdAoTQ2cAmmW2SUMEVGEWcDdzFgzolD7gi53Sn6ClCCgwz36wMXJVuUTMqn4KeEKWcgSmxuwBdOOrKXOWRdOQ5y7qeHqnGLkA1C5jRqnfl6fKS0U+UacyrsAJnaeBJiznrMV8C6V7UlIkuD22AkzW1a0U7wtkIKBLGD3JDKvKGeIunpDjmbwGVXCSRL2j2AjUmFGiBUzeO/VJpVuUpyloYkUr+RKKWi7FQEkXgxcggjk/qHWzjMngT6PzRrk1259FHEQKp92JXvrU2Liy0u0tzqYElNjgIgirEDiX086mGSsp7uBsAU9nac81IGtwhRAWhOvnJFIwpQzOIgRYptmAlTJgcQoavmupUtMjbIyMv49yjzOOlgaK4EFYRSDJxT4IYydZGIcyDVI+rVtLv2XIelWG8Rguu5fpRpnBX1M7TnGhGZgz3XFakU2Z5rMIOF6mMVEAz7naT1YG8wkue0YQpuF6rmCT0nkyrHjtD9Z/WODjOoQOlIf6pxQsukusnJrHs5uBkv74QIi+iebkB0YbRxPRIAIDeAMsSatftvEcJzMmZqXrddQkDdqgo1BBTajVgoEulx74jHEkm8KNmipEy6HWKqG5dqV1AYAxYH2c2qNmzG1FLNqDkq6iskrj1rg4/ZIIzQznXVHArloPuQo0y5net6ciDaYd+QyLBUxMXIBiNhiayPMo1zxrfux4wMUZ9Yu0KL/tpFzkKc67XE/jiXZefZ4iFOqg4ALhSVHaskFKWBpLtNGyeJeihjrCBjwWWUPoSPyS2fkNqWH9BIUMXBUeTUtnwJJWcmdCBhGCmW3rBJySQjMYUeASORsgpDFVBovn/aqTIRxQJSK8hp+P5XoNI2IWnOlYbQXDQ6wTGhqs4NYQubsTtBRJIZritC7OeILPf/JicwqcuBuIYQk25KQGbwqIjTuJFFhVddPo4E89i4FidbnOjZu2TVx5hHmowZHtHB5gPjUK/toPaYYRF9hIseAaqBLfHoACpymtSIxRVfneWgRzFYiISBXPo4IwnonmPXfO/AWSFGW3TMRqnmWK3ZuqVTKUU3/Y7QDRT6EjlgtCAjRzl2k1FZ9oktSspsVH1OHj6fMsQSdJeqAN0TZMNoZiJwkHmYi5bysEWHzIapSlPnVOhJgOhAQm5hqvoyEHMdDwwwZKKy+NjgA6fz2ohb/lrWWQTuoGTq44mCbljZgWp0g5EVyb3JiExqc2nobJAjWp6PI0Dpd3S/0lPFbIKGdPWMVFxaEUdbpEymyUVyn2g1Q4Vi6UANoUerVjMUsaomyUHmLmpZFMOiZIOS6fys0u1QIJk41Gtp9lvLz4rJXKp2heEFimk5SrfYmNQJLTaglQmASHqg54G464XRctiT2W0bkrFE0jpKtkjhZfB4lHlUGXLN4yPMo/zTNH7/8sdn+6dHx2/WzH1t5uLU9ppLj5KTBASqIDngUNp4EZEoFBuQ7nPBZVXY5GPS5eIy+g5RT7KgcngFGVucFkIowBAqUEUgLxvo17nIYVJhCp6OLKr8BEKromAgW0hcH0bMyUEOJSw2DmAjTipKpfvroxSCai6oKEPsIVpKTsxmY3WkS2eYFFb09SYnYdYs3a3PHImYuDRkYSzNLG2hRWZHcHRf6yXl5bTf4gTmFkoKUo9usEwyDsn0UkfJ4yjshkV81XVSEY5zJC0ldpuTyZx9cd887HSFCJun3kBJHmqkwyUIW3ZfBcQ99IvWtrVBB83JWMgwKg8W/UhKHTEMY0NEEl0iuSEsZRVFOoiTlXn5KNM4abvsufb6tBZ9msksYQa41XbU8UQxccYGuD/Xsh7rDT54UkBCVyHMW48FEloxCkOQXNMzASkJh1CooTJSPZYasUFImdv7uYUY6eNfIgSrmWOgV9u0+lKSIKpYa1eJ3SdZVuDcFhk8WRw4j+M2W6EiwoZKuC8bHHNGqzZVUY68tOwDGBGZjcGGEYONkVU5aIB7dULKgVXFkwbKyG9e/uF/ycf3R89e/jibNNLtfqo166bEUBzpDlXGDdWrk1BD5rpfy+PrdOTlaJibuTQbLcfBw+IAsujHQQXYSqTZcDUfxQo4DDcDLA14i425bV6P0ebdMTNEFXcqSK06VI2iq6mCDmQkB8ay4ns3+MDJYzd5eK+VAlbhp0aVVECtLJeOB/O/6frQtRFCCiMua9UV2SJkMmm2wH1oSUTIxA1xq5hgd5JYKWdqCEsv64hriWwxMkeJeDip7VRSlB997CvInrmv46okqCCkLBiAGMaWtYyoW3zwKin1GNNY5mrlMLg9AqzOdUQ0OhypaNTulAz6Yyk5oGGxS3k5azYYkUlfDUNvyKJ7VUyFqAIIQGNcPyooIRXEIasu4WibkDJZod+LrikNgYkKsziS4Bk4diMHjlaLx1Gmsfkvs/YGJWmyqEgi92fSTt/T6jiWCmL0kFwdzqlYVrIDFWi7aLTIeJiM6WKbo0J/Vp0ho0pEjrLr3FZ/HAF0PDqwldiMdiv3aYuQOW3OloOf3Fkl0mz9dgxQFGnjALo6JMQKIMEqP34AG3nZ7GZnTpaI/wjTqIL3egCnZk7y5EYa24aJIRRWqcNBc6pHs05aCXtAByocjjLSy1/7MB9WAG49yVMzN1m0J7qbQ3V7iqo6WiB5TOCKptX6VFm5VgBVYYDTEgYO4CFNxuNQj8ex/GcxR6Aj9CY+NfG5CJuI7CjHFXBwCCGT9buH/UsVk2gFvB20Xj22xSOo4mJGLwNY4w2i/oVkxpZvn/94/OeXPvMnz38+fv4XvfznT8cnf5gdiSa1RmjFuMuOi85sIHbQfYM6nignS22sQEZb0OVx2uKDVyroY0zjpDk99W3DihKSlc2uAFrIN+xKtF0mBgeYVrjMIWTAeqZ//zTibINBiNJLBFtbCyoNSMQ2njNJLnpmGqCAS4Y8hI5JYV7c7aAkAAcMHBxkr5Zq3DCLRawqKCHJiuc+hIw5+TEVl9vN82PJufpPQxhawJkoG9aOs4KURmcLc7UuQh4kZNL4nRK1DhbWBEw4YWwIqfQbIUFgkoZEaFCy/BEblOBsTQFsjU8hxmI5uA6Sd5yvw1DYWkEaKIOOFNeWtUFHnuz4SL3ZSOSYGUoQRxg9QTc93elZnsFCmypQtWw1OT+IEZ5t/4K9/QvFEmr3o4oK8337F1Z9IWNDFEfO9OrFucUJTfasEu/spsJT0HcEa75gILtz1YYpFTDXqYE8TP/6ctHxMB1lMpFReolILlYOWhoggdHFGaIp2g4El9X0ADZkbsPKYUQhW/dmPbzJEYZWQMjikwWR7QxxRFlWTNQBnOQwWw4697rPhYLpiAas4WQf1i9g1T4rKPfHx4ri3GJjTuRtKrpFiwdg65LhoDdbyjuVcyFSCg7MJbeqcx/AxmTFjdalNe4w6IlRVIoygCHTGMeci9JhgMLo7IPLj79BB87JVmUkwZeobwl6ClXArRk97ihhkSwN3Kd/6S63+NjgY9Jm0nxueojbR1n5WgNWSMuHU7JS3OaAU6AKoSzZ6utk5FXb9jGmkaa7ufV62SXEFHPhClKMNCoJ2KcBVMCqb4/grfVcbxEypzMQ9ahkVulULBi8AklhRCungOZBNWAd1IfRaaW1bPDBk5u+tQxoEUdFBVNzRzlqtTasezpHZkwVlCCr49UBdJRJrx53AwdFi8rXWa+gl94NOz1/kYy0Cmzf6vmla3ls8TFp/yPsMpGeGkIWF+koY08yKhzYGmNVwGWFchxCx2wDMiyNjkLCaIURDbSAGzIreIK6KBRgKLDIOICMuSgnfadmxdC31CNCl4ABPb1jG89WQSUxOqBwf5Svqu0P80Fh0n/UelqpCIWJ7CSxq9X680HKdr7Xa4FOxcoV3mJisp1SKyi3IwhF39RBib1aFsUiRRVpBzAk3GVl2uRisptrdAFXdromolSPEKlmkdwGW4cBUPWQCng4uVfYwSYdOCfgcuzJVIKCXApVoAJuH2ddFcBWa0gBAo9I+xJWpL1OfJ4tKOGBgeZrEBWMSkkVWLPWPl4IwKysBtC6dLfiZ2sdbNAxbWzFblWNVuEyVWC2qTacA1hH4wo4feGoW66hDTrmRKcWYFPr7jJKQTGgE9+ObNxxyHqAU3BghSdW2M0hdMTZBLnQrFIAoDJswoZ0gXRzVcxWorEjGSVgLctxUfIwJXOWEBqtuGp0pq4FbggIxp2YUU8QcIQxjGAPXofIBieTZcRjDGn0CM0guggcSOy9Q62thNWNckAje85KJy9CHiZkbpEA9wiDaJWIbGVUQK6E1/EMwdZFBRjGsb6OkS06aDlRH2EaZe6pbsGuusErCpAKOMBWDFMViZxKsdbEBiSksBqeHcRHWY/1759GDnOmCGlV7Yu12sbQALcG2zu2OsdibbcVJLjfqcMKPtogI042uXQHZ9lZHR0W65ruqBTsN6xqifWlqegLRmJZsswWI5NFSpov2qrFFEYgc31WJNBz64j0wxJKQznCitA7iBKZzdmSnrNlKdamVTkCdzmkpzsV8xMjogOzmjdGYGlgG4xMhhSn4HFLpvEy54K6jTkq3urMbmSxRK3SkMio0L62rQ1GYDKvUTwDwhwJUsgqsRtgK9Tjw2wKsh73ClBnYAQVLzI2yMiTBoreHwiRa3qQghJaHL5F0mRGa2ht4ItyySuWaYuM2Yzf4C4HswrphFspWAc9j0vVNwoWTgMV8BdFCuJyOmwQQrMyL7cO7ilEa5ml116jT8yECsVi8HXNWBRz26VwGYq2mJhs6w5tS7IaNsEMdtEBx97IFAXspxpIwwG09qptQtJs3GXocZeWigV6AFXE1tyo13xXqa04+LLRyhJ0NwmZjNsY8RkxUBIkU/oq4mbzhp1uYVYAtziwnarvWYuRDUZ41vkj2LpyUUawPu4VqC5IbTyXHGpcpoEUZJiwVnrpFiGTxQnM++bBAoCxgKXhVcDejiJaySHTTiQ6KLFn++LSPbbomAx4Si2lSM90IcQc6pWon+jFGndJveY4cktpBQ9scDHbPKf5qS1cRlcE1taOFt4XQrO2R1FpF9jiPAwhpBW6/3U+yuogOztzk4XlUEZlE4isH2aOCkMlEt93a8ymPxdHEUdZh7iqbGxxsnKlH2ca5zZpZBrd7kVKwJgbyuX+TlbZnlNpqAxjduJlmNjghOZkSiuK4TMPIIFYz1ED3GX/2qg6YqnjFLPqYz2xZ9GxRUeeTCbx8NPq8ExkFRENcCw4hoOl5zpQAbOvjrTEyg06OKyN/xGmcbaqHrpjwAzQJBZ7nRoq7m8221s2qRyxIRplMhItAf1hSszPOFcs17uNWlVcEWCBCsxfOYaRLbWtAjP5rK4YB7Ahs7lTPHKnKJIVcqsASx7jKv+bm7QCE8F6tNhyD2wQEvPa+B9hGhNOdrPvdQh1Pw85q8ZaQUskt272mLNFRzrIdN9jeu36W3xMhktY6ega30V6+pIpZBW0SnnpaKcagLIUoALr+r2MZAewwZOmHrw39dT6hOAIoVn63dTDbKVsHdEIfV8xqVuUTDZWgjLUYdOBWYWkCnqhVRtHgGDnsIE06lysYsNbdMx5WLh0xxYAoIqpYkDPi9wLe+pS4RigAf2Z4aNfBWC26Jj0CDP1FH+kpB/JDaBHctt4Lqq/legAZQQ/0iJki5DJdECde1fXYrDytSE4aNXc6jAWseLowapgjIZZuNjYYmOy12QoPVuWEHQLMjHKQIIeF4lWDDowO0hh9cw6gA7Osz7hESFv0fI63Q2Ji7s1pj5ENJuro4w9ggWXVr3FyEqXfYxplDWLv3MWT2JYj+IjTOLsgdfa5Vh5CCxJFVUH4mKJl4egki2z2wANM3JaxZYfpiNOBqQVr1Ka0g4T6KvMDsCrQ1i3iRyzKq/sAO975Sxv7sN0zLY7YO9byztmTpZAaVdxj5cO6ncIVo5Or9kYWUli2zRM9yF0Y3OyxPoiuh64IcsB95xW80RaGIoDGd2y06qassHHbFZSbA2ELcKHq1LqiELqN0CPEtVbHSlb92aEFen/ICE03dKLWu8uQpvr0oBIH87g5c/IyrKUsKosH0DGpIemlX1PZtEsyUpuOcLY2qk+NctzCJjQgVUsdz7y2qwepmO2IELsdLDJsTVksaLswYxWEAFsseTkgGuJClsiMXL6wxeUPYmzhfgLeBqeTroKTTWVWAHkVog/BwkIKTowmpYl7WtMTHa1u4871yMmg/VLc1TScGBG1TKC1KPcmtBHHo78FTD0MCNlLngL76v9qXKSCprjuCJqpTLtTiCr/c4N5cCrUMsBjEw6YqK3nRIr5gUUOTug1IYtLUwAgwMLt1txLl8lQ8JkYp70xDzMSaKFrRpIVp3Ux0Es5A5jBTlD363yYuNhNiZT8yw8pqp1wPpeqTRgvpY+HNEsIBWU0a1iUfEQFZbtOBdPMRKIIVvpLnCQaSQQ69Giy4Id3AcbrTNjg41J9yR4qodOeomgx7Me0BWp7NTr4uk5AapUsCNEXP7ir7Ex2zaSc28PonMtlsxdQaY+btWUU+1vZIBp5KXlJeA+RIaKOcs593vm8P8BkO55OA==', 'S2_304_GuideDesign_300bp_QC.csv': 'eNrdfcuSHNd17fxG+B8817F43g/PUkkqxQgaUAApWx51NIE2gQgS4G0AkvxZd3wHN+L+kH/Be6+9T1ZlkVWlofNIonJ1VhNAL5zHfq79X//3/3168/z09OHhD9Mr88OX92+fHt6/NR+f3//w/sPjjw/y5tPT//7y9OHN04O3Pz09m3dffnr88PDm3fPDux9CNT8/f/zL+0/vP/L3v3n8/Pjjxx8ePn1+fP5sf/Wjpw9v7cPT3978+OXT+7886e+p3y9fXPmO58cPb/WL56cfHz/TL/vw+eP26775+Pxkfnr/4eHte/rV+E978eHD9z+bxy+f3318fnj79On9Dx8e/vr+w9uPf6X3D0+Pb949fKJf2rz5+NPP9Gt//+MTffz53YP+G8Fa/ra/fX76wD+P/jnePX56eLEsD3+c/uWBfufnp58/Pn9+equ/sPnp8fObd+8/EB0fvzy/eTL62z5/oV/8P358/IH4ef/T4/N/9t+E/g4efvfN1w9/eXp+/x/vn94a+fcetr8O/b43H98+/cP/8n+j7/+tc9kAPTgzz+syzfO08j/TtMzLOhn6e3LO0Le5EkMKilKMinLt74oN5jfm65f/9uL1+uqb6V+MC83QD27W5y9P8n+vPz/+8OTa5P/x//8fn+w/EqX/9u36h29fMEG/++PDy98//PHVy3/99vW3L19M3z3wH+O7l4t58dK8+/z550///NVXz49//e0PROyX7798enp+8/EDMfr5t8T6Vy8+0jr58N3j94SY/qfPD/Qje+vjVz89vv/w1Xfvv39msr4Gi/1L+erBgbzf//j46d3vn//zt5/e/ZIfb4ibZQYxxMyyMF7u8hNOKEfzT2Z+9fL1629eP8wvX33z8LuXf3rx9fTq3409PlG+KlHBLMs0rUQP/cOLaJmJMhCVTfG+RCKMn8nrszU8s2tE0NkCKnUgWqKhlQNW1on+R3+0eVnWG7TQcpJnSkTLty9ef/v1N1g2I6yWoKwkMy28mYgTeqx86NDpw6x4+ilLTc62IMAXJ8D1j1y04zKTDagAPXwgrysfP3eYsalTNOKaSUpNY2r4FMZ/VxzK66pHcQ2pOLqK+Bl8xNMHK89kB6bFWbqiJl4q8zrLqbvqkrnGS5KvfeGrezBe8saLMxOfL2TfrLxU6EEnMXgpZKTUEGsOCqoTkF1VQCbqP5k//VGvJE8bayBmvCGTj+9pOnvpQlpp+dDdfZMZWiwKYqFFszHDltLhmSmdmEBHzMJnL1/UfDfRaTNtRkykZUDfi2f1eNKxK89W96y0gVghG4aWB6+TBSevHDc3WCnkAuCZLkw7l8oAvGzEJMO+E3lOqywVdqTUeTKhevpPiAJSVcC7UEBN5ysmtpGIYSsG5woRQ0sGm0mv6qvEkDsgH9HeOd9KA5y81XZiiqF1wh4AHzJ0VU/4EsSQw2jJ6PeFrDlFdNQKgmMpiNzJc3aiG4meajgUwdtoYUNGubpPT3AdJbdfPHEAerpvQBuDlwzxAnOP6OEvxDmgKyc0YjIp8FlAS/pRK/4iTuPzQNyQd7TwliKCFr6ecFmtt7lJnRsyhXfWXghXmAnMjLXHoGYLYTlaNohiiXsgpo1Q403MNSZfi4BYsgCbm4LazrlxbYDozIkaTwcOm8GLHMWwg+c71CQ6pgSEut9S0R+eG9/PYh/gJIi/zaEruN5L31A12MbmHiPyL21U5G3pyJfhvMsTO5E9BbH7Fpw45EnN99mJGzu5DBeT8FsUi9xlWi4c7KQ1xJGsBVFhpufP5EaSb9AS+QaCGpmLHbmO6EYfN2p+RlQ2fGlxeGLiHcZ3mLrit4gqiFMAIVAx2jrqkQo6QrDHVrgSKy+oWdeRs6YE+p04fM5PdsTpGaI+c73IKgzFSzUTm34rm8pYOPz/6w1eHN1neEZ/biDn47ud5Et2VppZJBIKA5ltQDaR1X0IIWfnOmjOA9jKoRqAsPM7fUnHNwJP3ASyj9m+4T8Uh0PVxbrJjXc5Kqhxb+n45AdgJ8TOjkPaZZ41vMWpqbVHclwL2aUmgDjpb1JSUHaRnAEComfEIFSMO5zdcZjKfUtdI4Z8TQWhjXc3hS3hnQznWziCztFR+JvL5o17Z0uCMQzEYRtFtaNs4/6K8kPRkw1XS0zIM4hjtaz36bF0Xym6CKnbAa6q2P0H2hkTWOEU+IqABS8iZSfGWIpldoAyneKKalVU7IV9k9NA9ESLm5zvqRnJOwH36ClkICkKaX9htQFc89TDFtEZDldwgopI4jj7sqV8EdJKmey9pigWp4gDForSRegil5H48exVyfqRqpuFb/e7/AS7Ie/Gu7rSVpQUzIRjB/kZPnpmPozATzIt51gbHbcMmo1JAZ3RAurFvTVAsvOMm2h4wXAged1OZ7V4rnFDb6oCz/tq2LhF6XYh/eC8dNZZyism1BGoY258DJFOZY6yAoVsFbkSOmrh/HZvYSRyshEfgn3yCWm+uftaN8jJZBQousyb+9IGcLdqzw/HYiT/SesHp5BUujFBxGGoJdpYBSSfBYSkb7jkYrij+URNNagV5TSE7DCUodykJtN5LiBf1mK741/rPYIRm1lgCEqcFFahFud4+vFTzcnSTSQocbETULD90+D9cAmJraDWmmmRvN6MC32BcXiHnMjXlaK4K3grxy8rCLZHS1MwCB7PUpWOIvXtsgp0J7lm8Qxc5kfP6OTrGOz+HK5pJFoihwSREJZyWqnDvk6Ls/reu8FiyPSjdVKSQboB7ic2EkLsIIVL/RptIwWe9pSA4IuCOF5W74ybbLT4b5lPxUvLHW5iVpDzeMfvqaOBTJoVpRUTyowXYUkPYBuZEPI4BbnS3/lWFQV3EdAZoT7njJ9qVph5kglexbma7/ATa9xQ9vso+wCWzUZOM1KuhNubHSn4Dr3CtrmcyNwDyJaOGLypvoM4XqHFRk1GJBBJ8Um6ytDncJuaTD+9AG93i2aAMn66jTs1aHBAihNpqwVVb7MmgelndT76khVF31HYPg3BDrhyNn5Q3MXLhpzNBX2tPYh8kx/HzQ+Cajg3igcIUoTTxgoGrWVs4uC8Waae8uRcTE0pZA6RApUNxVQ6KvtGkORHYidKaT+sv3kj6i47MXcEa+fsuhpg7cQeX6e7FxGbCetGqt96/JhbzrJ3xeKZ6d/BM+gz5YHjo2cMVaPtm0gKSwZrXW4xZL08R4xSpO1MbgYdM1KCIuZy7291ZOklbpgvAhJ52wI4cgGAEp1RuSlcwq51bhJTZxdiuc1NtkGBDxf+w/ELtcPWUVQc6iRnyZVPiOZoSD3R4VoruQeto5g7CtF1lN14S+dEjzc4X9g0BkunpNUtehJfV4JwXZ1HugYIANYe6SrBoIh91nJtrrbdWoG5e6haPrkZRK6YI1Aa52UA6s6HyHkkYqIU+S9INWwc3SYmts7QrmCyDOCQt84LOjtnqe+axb2aJi3v9+QceBZ4AWgcT2WAoAVA88MZxhsx2SDuhwwVat7gkt8mJhQFae8x1OOvmLgFuGj5q6uJSi4+hSdVCvLWJPoByE9QUMkKZhDpvwqyvcjc1TgSOchqog5H8r2cGJdz5io5IdE+Ashuv6FsHYCabUc1IzfSLKpJ573SHKUIyUXHFUpAITZBodEhLu+sTbsOvQFuqBM9dPmKzceu1CL5u1kKJW/RE33tnzIXo1l+cVN/ITttQpCLK3CknnQXV7eV5RkURZY1AwpcaikoD00PVwJy3G9GvSTMY/XDb9HDYQpFdOPtS3GiPX4pTgzdc6jc5omtxZfXinql5aTbFlrKxeJZg3zdyH3As6bLhvI2wMrZiIkGPTILDB2cQV2J6wox3EkkX7eBexfjFuOqaICYVKaB7WV0oXVtxOA4vZk2xJF2oMKV24J82assxZHoyUYzMtMiVdpwr+7SU8kQUhT32St3fJ88NtfV3IKBZtm0aq0tnIlFa7Qb3Ushk00NwM0O8oa2noCQz1dOHIoZcso58jf3GwuO1nqbmRL6m1QuLqwBTuXWY6QtgRvJWYkOIMcCpWYpRl+iJd8dINHuAaAdJYBLAYczdU7U8HmjfKhM7dot5avU0C4Uatx4UoDJdieiFYP1IkqAoKgbOeQvZJtj8RUgcaU13lS+pQAG1LE4o6YaqTQ+C+hoq8w1aiAJA8A9ecNRsx3DrLY0reJ4onUagh891NU4u0DelyBWoBLELClqe+Wc6schx/EP0r2GBcoMOHXuskOnlCCycPYJ8jQAO5uQmWXNJeknn1bRxV40TpqMKy6TX8mdrkCFs1RAjegBqtbly477AfQIzhnyrDwv/UMrbvS1V+fcYohPY0UoeztPWR0/7oWNovwEVHch+SCWIOTnmZ/A7fXFena4BUFXCIhVqhS1y8jFAJbgOUHRzFMXJODQF9ia7hDkmttQKHtlswHO57DNdbDJwFDm66uXTqo4aTC0e1rhsiUGtKW8vKHLXkHeZWryWMxk7vCEOThJHgLW8k1mOFIhABX+owZ3st9IKgalXdI2wxc91D+2wRe1+ZR9RyV0FLdPyafYXfBtKHqqQW+w6AZiNS299vYWPYGPZ0Etj6Ywnjf9X8cCBchILKcIhrrp1fhKhGRyygAKFzcxqBx+BwgXN1cbihuMlNFpO9Ao+Hu44S5GAO+HK9Qpm1HoyOhZRJ0KCsnciLVOWx1TrMFHlsNjFHzwiiwZS4pQBTeWR3pGD5s8yGRNq5QaQKrgHj3seeo764ejp26HskuS7JtU6BarZ+0aZ766YNk8ZoDhKQCsoC0gu5G54d49TkeIQp7K+d/mxgaWawDYqzXU4xez19N5zBqTbOBg7M66SpjndJWnUrjFXlG1tb/jWgP9tA1c1H7GVJVerFXnzMCr+DuYon86auP1x7a28dPYKJyku1HKeXqxAXrVomMGFJBnAcAdjgKyH06x9MSNtwYBZg68TyIIPKlg8g1uclVQL5pj25FjPX+Y5hevnTV40q/jjGhy6hw0lP73xA0d3HQ1FQh3EfLcJqzvOAoPVIcKw1+y4838K/TcZ6du7NTh2CFnUeiBSPui3oSMi9OWkWJCSC00blokEFmEXAB7mgDxooDHu3GIiWZjA/rsENlebxNDbqeAdBF7D/74xDQlRirb1UAWD6LbgRLrCmxHA/imb7haUMBlmXKs4zDD+fNVhjlJqzBH3e8wE6syk+wvCriPT03s1BSDZBYnz7k/BIVxavgVw7aeD80K4GJtgEr7SoD/hV6gH4carmwHJ2IOw3vQMqar1ASuFAQIuxRxOv4B3M/fZpA7n6V/Bpm+s/goeZh8yhZFJVtFcBoE+TrcUdPJIesGbVfwpRa4nWzZ3GUnpdhRGXOqtPK0EUWm8dmRvMgQ7i7GmVuNNfBRzCDZICBn/SjXdHHwZD8QN95ADW/SGoz5TLP+KjfcBiugul3y8/gnD923Sgxrk0J0XIq+FjThy+4KPCfORXIpoyLWeFOU+qexXOgo+jAQPSzMvspAT+TO0Zm13KMn1JYUNbdrOPLp0OKkQk/LnZ5ipCBlQSxwljG5fQI3bT/n6YQBCDYo4Mp1AfHS1hnADDxxw1nPRdJWEiXdIhVXufE5NwUXk+QO3TCszGybqo9mRNxvnmUSIVYN0WBLzrVw+SSjZnnuIN413z9tMQ8XpdjYocUBgXHEKVBJOXd10hvsNBTtCMoXtV7u0ClhsOJtv8m9Y+EPjlDALVdh20XoaYVubx5HzsC5om8aXVECWhht4Zwx4yFUv2KQOwe4xMm6yUx2sX+0G5UbD09L2PYT61mI+OakxZM9iON5xjb99JFT4kCpWEWsF6Ko7Urf8kDUREiLL6KfgwFgy7TepYYL3hTVi/6ZVo7PTg9W+GTEh5omGcoDgrppnHzzqWYBmQdy4w3PYhSQ3HBHzYmabCArtKh4DpLlKvx7nRqrHzU/3inctiRDgPK6yihi0EPXtiVqorW1ePKsAbjqWN6kqKDEIbPjlyRFs3TBQFg6KNi+Q5Kj5Scg7Yfe28M7nLRolJgifbDicM4YCaaXFTlN2TkenkJPH1CwRF9zjSQ/23Am8YmUapalF3LJZOXerHeFFC5l52ccaaaekuI7K81MGkznFbPq2Ditb2Odj8Z17YIiF/IDee5QExTruOxEzHhA2TFydqtcUffYcdV1dqwbb0P12ETkjisdZCBuAjZXP39DcDz0SwAPUgbwOSqoO18heIna/P7xx083Ylov/7SCy+lP6x9evnr4+pvX3y6dqlfffDet3/7rNw/ry/+5nHkpfZNUOTrxlx5ev8oZW0ECahhus+XuR0R2sdD9OYksOyoJtjkQlWWFyGcASGQXA7jSFLQ2MDXRSAM1FD/6qIPlNjU8o0hAvtApPX4yeHM+ea6VzLKC+yBdIVO3AVsj18GmKoBrawXQDy/A2+EWzcYMj3+A8C80dZblpMN0nRlytBTU4S6u6HuFG+0QFLdtc4lY30xldKrBeEoe+wrQXBRAvoSCmEdbNGfUeLOI2m8fNgNB7TvUcLUOQPDjrZruQKRg1AbkMgLMUuGIDlNDhIRioy1WQKKjF8DXJCBAMXBUZmSA56rjeKQERYorrjOT04mi4dZMN/dS6mVcECUQpZheLeosS3qw7ICgaK0iv73zIQ/MjnTIoGe6T1bWdXOLnYRUOVC7zP2Ww+cc4hYfTYXV3lbReuudMmufiMGqx8WzSgNQJGNRkI39U54f/Juh0r9n5HA4R7QCIb8uFvF8lxyeNCOohr1m9OGrBmLscYvUNIm3SNoXw+PEy/yz4QL9KoMFCbBUF4PSglWQ9hp4xy+RPBGTrZEi7EnayEWXfr5JTC1OQdsVcHl3/KOYbm4lBuo5kmjA6BR0Laq4EBl4qQTueQGKLhZFXlRiGGHaw7mTGQdix5sV4a3eDCMVJ/fYwXB7QRhuf35RpQHoKZ0eDLcX5U0ZyAhd5D4KN7tq2UUQVHmEBlC2G9qrktY0EDfRyA0leZhlEqf8DjfOtqCIO/T3okLu8MUD3LKq9CSea4WQKJwqCDCt6lR5Wx1dS0VACvrG88wwgLQbFjfAljrxknnZgA2p+MPJfJsXy+pvDJzd6cB4e/wFU3uAIpNRrKVb0he9bDc43cel8mHbFOQWBLDMuIAwXuzmRE1l+cRZhr1iGjmfxsttakI5++hsMx2fl7YdwU00PLSltc+y32ShQ86hQgyaULSc/sW7nPun+aJ0//ilkCd2CjpjJKK16Azu3p54i52SUkcl7SspDu8wpL6lyPSXEfa4nfhERsNZn9ZUs208TI6Ac16ALdUKqNaOM5XoghcJE4uXsELJoyu1XuUls7QJwF7aZIRmhtQdzMJySqtOoVylJEnvbW4RsrRtAnkTADywHQCxUIC6a/M4frzmxEvE5IJlVXVf2DXrHV4iORYC6nC1fslv1HB/uDQvQFBgltSCHsCRt0tFZV9u0aasoDUBLIGycw9iGIgaDHRgW09UNdWPuklNaHQxCSjDZRZS7A4CDz9DZdLS+1rnXqfvoqmOTBYuOQco5EICsASwAL+b7+VLOv4RfKKGRZNknqsEzU9TXa9TwyM6BdQBV82mKtDMKlO2pa1X7OLeNNVaya4VABZ81jdV3mRvy7gd0CeOeP7ZJIFhme960mG9ylEMUUGKw62e3PO8FXG+Re5v6bvjNhhVlKrWeZdosTDwTt94v4FwofHCivbjcIPx9ZB5mbSlYWuBvsqNo2UjIMWBRkkrNd3D5KlVXE+CW1zSUpsATjCFXIVmgxPQvAWoNQUFZRfkGyBPd8ZM7sOA0aW5oP9lq177dWYai8oLGK8wPZXtFIZ9gyQUxJMwqHObr+0qeQlcgCKI5esEZQiTAaXhRMjO2Kmi7SeiAhAU6ON1brETi+uoueHWTt3O4mYkgYnAOd9X09ZzR66lC4HMPC+ABzMBWFpx8pHdJaNCGoeYZo129U6S44UYxW1iSktRARRwzubHHP8krt1naCyfDmEXGQEHZnrbs6MrumWeTCAoxf6OC2cVjdcUfsaONzIEeBY5IFE0nu+xE33qKKbxjpt+i9Pf/DqJTBvydVKT1NdOzbRvyL8G4CJiBV5B9AM7VSeOIqbYa5hLlKXmTa3j1znixKaAVMdbPb1Ov3GDOMrXem8i1Iw1Mlpq5vpqB+BtFOD4cBaQ7d46dgMxkzFuCOtGpBcQBrzNTEWql8F4qV4umFFqikxt2KpuRNJOBuqQ6Zd5Jh4/eTgBP2kfyXu7m4FSwkCcIP29QFh+EsOGl8wNTsgaxnNf13d8+agcNk1Ibu9Fv/Mi+QXMK1Up5xzI5q1ZQGWteQHbm+pGSy2cmOHgE5R4pYNVj90+1/YaNRsIF+roxz9bthZVzt8vsIEnaU4V8eJJBpTSNnKt0fVDgH7L1AGZfwKav5yBNxI13qDbcEHuG9JRHAS9QY1rxXUQ7GD1w9xi2pnh2LCqI0FgFd3gXeemxRZLpTuIAbmWTt5EfVOS3Y0mJW983Gbwc9KiQfs8Sm5W6Snr/vhV0gp58fKR29WXlMNX+eV02mjJTFKPLkNdZ1GHVN3D0GwNlQv0CUSOossbriQBKDvdwzASMeiZgjIQSijQ+yyCHVeJiawMKSCNZxSntnFTDIZpQ4dskkp0TTRkw2mXUrgqyfOECqeAzhIF6AUfS0f0nJtqxF1YRKtjkq7nW9zAJRfQLuRny+HLBbYcDE9Uh5WDzl6R2j+b01Ub/fQ1NgbJ8rQTvCErUEHaO+CtHX7RbMzQ/TKjMBT5OrSwLv1Ov84MV1QIRWFv7hz/fiKHsVPjjBRPsEs1TWLwrH3aeAiBvExWKAaq1vV3ob9L4xUKnNPjjVQTy8xEEaDdpkpepQc9voL8vnbWxuOzEzd2goyqX2adRoDhm33uCX3MnibLYwIFblllZDkBo6jtUzEDxHHO6EHLFFqCZlGXQqTrHj10xcX+qd01Pufjs1P9xk4yGq7A1E2UpC9boUkJrDTbqoDmFBSejAKwn8Ce3UjM8JTxPsQWdzjiO7eZyXT9C/B7Pch8/Iu8nrYUzxbX6ScICMJK3safsHgz9hEDNhoF+P4mDpfbPOemoiYSJTjc5gEx8PUeNzUJKPZypNvxzZy2ceMxKBHas6IIzlaOTq7wJoVqg69BQKQjGSCQkSNgQBvnjBtMFRcjcO1G8nZNXeGGezQBLno0eWD0wakpdotrseg1suF8jyMpfuopM3RR+xqtghKyAFaTF9Dyvns1jcRMNohOyEWFQoGeB7/KDAuOCcj5IkRx+P10chl84VHiyzbwbpZslXQ8uxhiaTEURZVNvcizAC1X4Ahq4TKA0waipxoV80P4Bh1DKgZ0nR565zcUB55fxhIKnagmw5ZE9FqaoXUOHus8024KmWuVgBqPzGEUPd/1gsKFkskI3TFnBAWrBScocsM8gp7eu0FQqFzMJWi8Yq7itkAp/e0vMnFTRmerHG3XekkpN3a3BYWcFFmefiyf2rpzsdJI5ASDTmiZ1iWKbadZH1fJSa2Tk91F4pzFPA6/ufxpcyVJQYgo9iw12r0IBUH2GqNVVL1TFFCiDYT8+WmEzuFzEOfksIM+iyOhyfM+IvoWObVs79rlMOTj3/Bhcyjo+p5FhEvq+tfTPItgSm6V/tcACqulA3DQQ8B4QpHn1FSDTk7MMlulBln90KvUBLroBKSLVXN8V+tUxoQBDqumyudVIhnd1XIuxgLxDgJsK+ubnBXUPLBNeCIpYhoeNGrn7qr3STFXSCJ/NClov5hpe/xDeZO2c5jigDo4MZc106dR0+Ss4/FLAnjkugAiR8AlOSWNxI03WjiJKmQoYnc1j6vcZFcUhMsRiu7wgh5lK0SJwSD9sIo27SRDzlYphXPeusobiH47Tsbw08YqzzxcNf8ZK9GI1NSqiQi0nl1nxXH+Ac99+sHHfHyzOG/Ri8hNDqKbLrXZnKJZ+2yPlCyH1J0AdlXlDVEjYM9NOP79fcYMC/0tMolAZfe7wtR1ZrjtTMBeSmmEmylv7kIspstfT5IC3saYcYeZjSmHkAQVF5u+41C6IPqj7NgJYSR2OG3Vz16kgDHw7R47XDkg76yL453DW6CdfuBJNBAlCSEJ4dPMdTpvYwYrjArPaWWULI/VFhQvCgJ9HCBUcWIoWemwn0VxahXNyPkeQ2Qib8hfDG4Nx19BZdtgCdp/UkegRchbg0gxOcVQGx01DFogjwpvOC8MUOy+ZLIcv2TynBvOfXZprlWrJtXrvMZNs7EoyGW4k2dTYXWJZ7cu0nYlWuFrHzRJJ26IrgYWGwIoXLTPoLLgC0CoFyEuPxI3EckZnDQiacwnz3ybm1KdcmMvMjPH9zXr5jnwqBjttlo1PDovJxMZPfQWz0xGAD+5URrPeJEWznEkXrKR8uMVA7SlfH2db/ASgz7TwK31XMnWGSoGs8wQ00KT9NJ9Tsdl17RzIpf0M4rWZn2XTsgP1xVyTk81OvJ3QuGFMnWXHrrIFIXhrqu6zdHhbl8pCkRF6TqdDYuG/rMrrXAkC6ii7hbvIvGqqNS9znEbiZ6t4xPFpF0z5x49lWewK/rFkCp7+DBgTdv2yokJQvv0JtWwnEpLXWvVRpcUpZoV1ew7qv6y4OL4wa9zhrhlbxH9u0kU+pd5U7G9wZANHflwwdDxI8k1b1YhbRrIh6ueONytZe5LKMBhsB6A52zrm9g/ysONKj3nphpVbmADCMm9bajDNW6K5/5pBsFepD5rPj43W1qPBzugSpDzEKKE10NgzdC287nGAFAcV8Txm5y8grIPDhY3EDPFGhmGjAlnc29uvMVMibYq8Jc1Ov7wDkU9+aAl9ALBRQ+dqVdScmomNx8z7ShBGOsFFHx/F+JePDvb499WZ/SwUv/Ua7ehE9hvq1v0sMmkKLX9mXP8qGCzW76mJCNzfyfRe5N81tonCLYWSgsA0fK2YcBFOgIuJZCPP4n9nJpsZAqwauog4L7epsa6piC0cZ31Zjd/gnX7F4zlVHUdxJhP+g3ZZR6EAcBS9ABcnCwg7AcRBTcSNVzkhYJSaW3UcSG3qeFhMgDlF3Py7OGToXSadnKqMxD7mKVYR+d8LX2ASnY2JSIQoKaswHYwXoTnnBuvw0sRG0SocMtkXeUm9Deh7eq1XR2Ams0QrDKPaBI1FC3u6qM5afP4HEsWUHMUUHwHMY63ak7URCMOp2goLpIEnW9Sw/aOAN/Go2YT/mDFfq3sknEh0Kc6iwligkoQFHkDCYq1KEr2IjV8/F6sc37QxTctMv1sFe9zc8qv8sNJ0f6p2x066fCZrBa2eBctghm+g0xLniR1s4UsOE5RUhDQMBSvcLFkVZB20eQ2FDOQ7V+nXsuELJ+WFFxlpiJczMAPl4doW5qvNijaLrOUBmI4kU7d4cxwCJH8h8YgWZ+iAJv1ja1ttBjgiZpmDQpJoaWDGmQMorxFTWyF1csY1H072gguw6kIrjmdBowKOI2P6h1uTczeJ46FAcTS9I0PCuJ+EGUZiRkeWY/bSSoEJ1HTuclMrFUBe2W7EJc7fIirbbO1XcMcK4iJQ6EKfZ6zBiqcbc1ZFqJiwELrDCzvMAFppy0ZB9hOJ2KikUHAs2jErCL5cYsY53mmDMB4I2Va7le3Z33bVfLjooKHfplFW0FyDHSw0vcA+BD1DetFA4Q6MjfR/Ao10z1uSudmPLOmbF5mS9IQot3AUN3UiYuYz1S9S1xyDMRbUVDkcltBreyO4jASOVmklyCGghbGaZlO3dJXyPEWMnBA4UJt/OD90q9YPIDfBykZWLT5YcaQh5OUGZde07kLwCENBoFL5QSUizSDDwe/vzdeohHlO/QRSedDL3W7zksNAvgQ3/NydZjgYXhxGbwknceOw6WPrxJrzxuWNKks2gsQyIABYHUvAXGfVTi4UutGipwucJswf3LZ6m6uk2KTkkIe9z4sfHRSMOD21UMxqqkpnvYEhV+slGJKKZ5VO/jJLgA/C93uePqxgsEbIdUgCiMjf1eJdEos79cJCeQ1yvu9vvHB2UDX86uHZmQU6SyDs/GUA9azFFtttDeKIhac0nd+Q7EMtkyUGGeNSooi+oKxkpsu0HVmMq0SRfUyQemPfqQkOWcdJJNW3USLNE0JNTyNNRVWH/OKUDwCxNoB+s6lQZP/J45QR7xiWhc6fUUB+h5HXNOlqF4oLfgcjm68oMyRuAnoDpdWMtldvZuMtlGOtH/I/BPQmoCC8ToMyuXchoPPwDvREjHqY5GpBMhPLhr3vUpLTvpRLhfiE+nonhGqG4kVmLoS25xlWEMvyPcsdVg9yzwLihx2EIS+XkHRDpUrODGDeYmQkxftdGymu8xwv4aiclG5l45+QWUx8Vj+mrfQKuMaJC15kq3LNTkU4AMllgJi5AsrlgjKFyWxzhU/6IiqE2tkGUtjkLS9zGinX+6xluhfV1Qv1lNrYVjSinDGo8elpkTmFkB/VFUZiC/yIiAYRCAkfRNC7W/qsGaQ0ONZc2CS+mKoJCJld5seH6OCPJoHioZ6YoVV1kXbbtLFc15uQ/ZACol1O4Cq7yjSra+ful2+7uAiXCdivHijmr7EFNP17yAmsASMUGT9fjzT0d306oQZnjkukkoS9UM56LoxE5uvkSuQgHLuyCOcDhT30yjD4b2JTk2UgI6IUiC+Nc3TfWpiStunPNNhzHO4SjTDJ26dUo+Lo6NSiy4NQoXu6cydDgCZBTUZcCWJgMsGoaPXzJ5oyUYVn9GCKKU2621aWAxFP/LjTJ18xWVHoAQqXFBBhKMlLVNdfyEm3xx3DzHwPAEMwNLNBOBcG8zZ6rRUCD0jSIrhpMtJUvQaLY4ntgoIu/Fm6fCkaNaSp+KhHnZaVTezRyyqCcU3OjkUpJgU+NbftGFP3c6PxxBO6T3USUxTDwJe4cezz8QgoMFloK3E8nxghcuxILiPLgWpxT+rbyTXknMtAClnAYUHbTKorg2mKH8ihiyZVQTIUBALgQXVsLtOjFdiSmyXusUHv5K8xLlYY78PpUfv7kn5kByA6kKyhTXqGDmmQ1Cgk1tQdHawnaTEZDPJ+aJjJXFf3yPG8qAhRWksEflXWcIOgZPevWNDixrXbTh9cXSYsIgUQOJ0L4PKMxMBLnWcffIHP2E6L5z7lvms0OPVnsvbvOSsvORaLoVtjuxd/zciWz0a', 'S3_Four_GuideOutliers.csv': 'eNrdk8Fu2zAMhu8D9g69j2ss23Hs3VQndQOkcZG43ZFQbNUW4NiZLLfdY+28w4C90F5hVKIW27Bh9wKC8ZOiSPkj9ePrt6HUUnZ4xTdQj6qSqCrotapVJ1o8eQb5aZRdKdH39lJDM+5Fh2WjsamDGA66f1CD6m18KYxo+xoHI7Tx/rolu8pD+VS246AepKvp4k/GPyK06CpnaNkKQ2nR9C95y15L2KsOK0XZ7G3/2MTdAcRoml5jJQdVd/iouqp/JD9KUTY4UGoo+/2Bcu9aSdumQXci8Dwb9mRkZ//H3aMRA66zDG/4NVJlLQ+9NrJyiWEvTNmojnD0oy4luLJ6pOT3raiJj9oL/fm5CPUALxZzfJBa3StZwekcvrTDxZV9Jd++ueLpehsEERwFhgxSXmSpXTzjWZqlnGygPrEAwiBg04j5J5Gw6CT8KHQiDuEd3N5si82CX0Pgx0C/DJeiHSQUepSwNaKWLOHB2fcvvud5Z4Qzvy22y/kC+W1xlW9wvtgus7VldXGDm8WKF8u7BRY53mzyu+V2ma/5CumSfJVnsM6hMeYwfJhMtHg8rwn2uBsHqcu+I8rmnDoxWfc0O91K7EjZlkiDhMH3/HCyF6qbrNROW4DzI9ln82QhOwK9bMXQXOrP50PjmEVh4JgxLwDO0yzjRWHZWW0/DloSJuEsjqOjiKdE+OgJnWc29aLfoMXeK4W2YVEC1s9mkGUFLeJFpNK0sEPGLS8f2CyKpyxkU6ems+ikfEL2rKIY3sM8/7h20Bib+a+eWgx2vor0+CKz1L7TIs3+R21Kx52KExq0X6glSfAqof0EbmUpZQ==', 'S4_28_Unresolved_HARs.csv': 'eNrtmM2O2jAQgO+V+iZZ8gck6S2wsIuEAEGW68gkJnGV2JHj7LJ9tR76SH2FDkuyW+2hlcBHw8VjZsYz30yC7d8/fzWppJTDY7y1GkVy6kYx4EC1jZUSRUqRA8ugluKZNUxwUlpFWxEORe6HkBZSVN0EGknldALlmQP0lJZtw56plRasqqEmPJFi3Fld5jqri/DJioOQLGe4JuQty2gDjP8dCVSkrhnPUbH7/YWpAnVIqwohoUY/FeNt4zsOSFoShV5BCegS+7ATrWpw8H+LipwuNpAKSSFjGD9PKRzqD2dcKChFSkr2g2bnkHuufk94AqjHcg6yLWlPu5asIvK1D56eSKpgMruHo2h5ZvU5dVqpEDJDNIpieShXTL1ajWglxtL5xsKyWvWTXQJfv3gnLPXA9T3rabWd7dbLPS6xWkPkbKYJTOMkXq4fYLrez7bxw8zqPs7b17JQcbNd7xe7xXoVL9/VN/Fii04SmO3j5VOczO5R0zpPoJvFfIFyoVTdfLNtSV4GOZapPbQNlangCqMfpKKyVwKLypfkgKMDcqEKsMae4w3tijBuL9lBnlO/f8uvFy8SuOcOhnlJmmIuXwdN8b7gZbG3FYgqKCfIT7Q2qt8l88nubiPFd5oqW+FzYJPIG7lHP4qC8eGQuqPs6Gbh0cmONCPHKAjHoRulXhT2HD1/ZDjq4Dg0GDVg9B2DUQfGoWs4auA4DAxHLRzDwHDUwTH6hPFxtZvCJF4khty/yY3NNuc2jI/xdLVznZFpwOvIBWaDqAGjNzINeC25yDSgBoxBaBrwKnK+Zx7dK8kFZvesA2PoG4y3YxyNzJlYA8ZxODYYNWCMQoPxdoyhOdhdS868DrVgDMzNvwaMkWvuuXRgDMyfsw6M5srhVopb1zGteDPEyJz8bmP4B6X0K3E=', 'S5_Public_Source_Inventory.csv': 'eNqFVMuO4kYU3UfKP5RmbfyiHxAURQyYbhSCEZiMemWVyxe7JuUqp6oMzfxV1lmMND+UX8i1TdMdzWiywr7UvT6nzjn3n78/71SjGTj77cpZcJlzWTjJuQZno3lF9TmFZ8psypSG9H00T2sNBqT98YdY84JLKghtbKk0ETzTeH6Qg+GFJIZpXluntLY2P3mepie34LZsssaAZkpaHOIyVXlrpSSVK5rhUyYgBZtSkYZ+eONVlEtv1c9N593cl9f+LQ3Sx+k2XQhqyoU+u6Z0oudacMYt+fLX0PdJVhMjVE2oVo3MCZd1Ywn2EOQyIcisQQZ9FStEKktqrY48h/YwEYq1DDUr+RGMM90nj/E2ncXzyFnHX99B3WT4caKhVoZbpc9X/j33/yHsbK+NJFesqfCODIHnGphFPGUxHHk1lYlWd2+YMKU0ykYtmAmC71S7/H3gAjpOLY92RN5oFJjYkhtEeeRweuG0jTbxbpnE2ydklqSLeL+ep8s11n9fRh/eUH3YReG9PwpHpGjwmkgFlubU0ivV0+nkSpZxV4rKlbx0C3X0ClDenw3os0cZc1nBf8Hfn6+znHfX+aE/qEBfhhvAJsnw6i+F5dwQikIWIAEfkBu1XElk3lUEzUDgCd3TZrQxONJSXYA175yHKO5022ARLKHCJazUqsIZkrSmREOYK5FccVfpwgt899a/C7xcn2nustvxp9NQ//HRmfUNJKNoN4yF1Zy1YMhB06LTbvJWnCuqLlEX6bDNiZPHaJvukv38qUOXlFzng5pqe0YGlgpVkJpLifqhfypuv+UqakuQtGp97uHoQbJ4vxtstPqI3vGsBvDoOLwNDsPx+P4uy1hwmx+CfHTw8wPk9DC+H92NgjELxyNMfhcA0yfBICO8rv+YT2m0ewvrAm9C8Ly0/MBBv/I8gm4rOR6/aItLAUC2Ijq7aBav59PtUzqbJtNV/NBRn3Y5GghuUDyFel+ih8L2q+lb2oSjwPsEUuXKDcI73x8GvjN96cNJiCA7kz52BHM3wXCdpFC0jfgRcbd5awHj+oMaZEtFnF/hoze6wFyWQZOjAps4idbJcrpKd/F+O4uc/frXdfxhjd76bUYOjWi3huVMwBVyXbGvc3E5ZDxsC4Lw5mZ8E3rOrqlrAa2F8INtuEhFz71BWzCdnTtqr4Fvt1e/mxl8h8x3gP8LtTMd9Q=='}
for name,encoded in PAYLOAD.items():
    (OUT/name).write_bytes(zlib.decompress(base64.b64decode(encoded)))
print('Restored',len(PAYLOAD),'independently audited Stage19B tables; Google Drive not used.')

def readcsv(filename):
    with open(OUT/filename,encoding='utf-8-sig',newline='') as f: return list(csv.DictReader(f))
def savecsv(filename,rows,fields=None):
    if not rows: return
    fields=fields or list(rows[0].keys())
    with open(OUT/filename,'w',encoding='utf-8-sig',newline='') as f:
        w=csv.DictWriter(f,fieldnames=fields,extrasaction='ignore');w.writeheader();w.writerows(rows)

cross=readcsv('S1_180HAR_Stage19B_DesignRule_Crosswalk.csv')
guides=readcsv('S2_304_GuideDesign_300bp_QC.csv')
assert len(cross)==180 and len(guides)==304
flags=[r for r in guides if r['compatible_with_author_300bp_extension'].lower()=='false']
assert len(flags)==4 and {x['screen_HAR'] for x in flags}=={'HAR169','HACNS336','HACNS643'}
assert sorted(int(x['min_distance_to_catalog_core_bp']) for x in flags)==[328,380,993,1172]
print('Local QC confirmed: 180 HARs; 304 guides; four outside author ±300 relative to catalog.')


In [ ]:

# Independent public-author source discovery (small metadata only; no multi-GB datasets)
SOURCE_ROWS=[]; ERROR_ROWS=[]
STATUS={'stage':'19B','google_drive_used':False,'network_discovery_attempted':True,
        'original_author_HAR_BED_confirmed':False,'source_provenance_noted':True}

def get(url,accept_json=False,timeout=25):
    try:
        r=requests.get(url,timeout=timeout,headers=HEADERS)
        if r.status_code!=200: raise RuntimeError('HTTP '+str(r.status_code))
        return r
    except Exception as exc:
        ERROR_ROWS.append({'source_url':url,'error':type(exc).__name__+': '+str(exc)[:190]})
        return None

def inventory(origin,name,url,filetype,comment=''):
    SOURCE_ROWS.append({'source':origin,'file_name':name,'url':url,'type':filetype,'comment':comment})

repo='NoonanLab/Noble_et_al_2024'
repo_url=f'https://api.github.com/repos/{repo}/git/trees/main?recursive=1'
r=get(repo_url)
if r:
    try:
        data=r.json()
        STATUS['github_tree_sha']=data.get('sha')
        tree=[x for x in data.get('tree',[]) if x.get('type')=='blob']
        STATUS['original_author_repo_file_count']=len(tree)
        for f in tree:
            path=f.get('path','')
            inventory('Noble author GitHub',path,
                      f'https://github.com/{repo}/blob/main/{path}',
                      path.rsplit('.',1)[-1].lower() if '.' in path else 'unknown',
                      'Possible HAR coordinate input' if re.search(r'(har|guide|region|target|coord)',path,re.I) else '')
        STATUS['author_repo_bed_paths']=[x.get('path') for x in tree if x.get('path','').lower().endswith(('.bed','.bed.gz'))]
    except Exception as exc:
        ERROR_ROWS.append({'source_url':repo_url,'error':'JSON parse failure: '+str(exc)[:170]})

# Check design script directly from original-author repository.
script_url='https://raw.githubusercontent.com/NoonanLab/Noble_et_al_2024/main/Library_Design/LibraryDesign_1_HAR_FlashFry.sh'
r=get(script_url)
if r:
    text=r.text
    author_padding = bool(re.search(r'bedtools\s+slop[^\n]+\s-b\s+300\b',text))
    STATUS['original_author_script_300bp_confirmed_in_colab']=author_padding
    STATUS['original_author_script_sha256']=hashlib.sha256(r.content).hexdigest()
    inventory('Noble author GitHub','LibraryDesign_1_HAR_FlashFry.sh',script_url,'sh',
              'Expands input HAR interval by 300 bp each side' if author_padding else 'Inspect input slop command')

# Zenodo archive listed on original lab website; fetch ONLY metadata, not archive binaries.
zenodo_url='https://zenodo.org/api/records/12600310'
r=get(zenodo_url,timeout=30)
if r:
    try:
        rec=r.json(); STATUS['zenodo_record_title']=rec.get('metadata',{}).get('title','')
        for f in rec.get('files',[]):
            url=f.get('links',{}).get('self') or ''
            key=f.get('key','')
            inventory('Noble author-listed Zenodo',key,url,key.rsplit('.',1)[-1].lower(),
                      'Source provenance and whether file is actual experiment-input BED require inspection')
        STATUS['zenodo_metadata_file_count']=len(rec.get('files',[]))
    except Exception as exc:
        ERROR_ROWS.append({'source_url':zenodo_url,'error':'metadata parse '+str(exc)[:190]})

# PMC article supplementary-material links; failure is nonfatal.
pmc='https://www.ebi.ac.uk/europepmc/webservices/rest/PMC11244942/fullTextXML'
r=get(pmc,timeout=35)
if r:
    try:
        root=ET.fromstring(r.content)
        supp=[]
        for el in root.iter():
            if el.tag.rsplit('}',1)[-1] in {'supplementary-material','supplementary-materials'}:
                for k,v in el.attrib.items():
                    if k.rsplit('}',1)[-1] in {'href','id'} and v:
                        supp.append(v)
        for v in sorted(set(supp)):
            inventory('PMC article supplementary inventory',v,
                      'https://pmc.ncbi.nlm.nih.gov/articles/PMC11244942/',
                      v.rsplit('.',1)[-1].lower(),
                      'Supplement name only; must inspect data and author provenance before using as primary coordinates')
        STATUS['pmc_supplementary_link_count']=len(set(supp))
    except Exception as exc:
        ERROR_ROWS.append({'source_url':pmc,'error':'XML parsing '+str(exc)[:190]})

savecsv('S5_Public_Source_Inventory_ColabExpanded.csv',SOURCE_ROWS,
        ['source','file_name','url','type','comment'])
savecsv('S6_Source_Exceptions.csv',ERROR_ROWS,['source_url','error'])
(OUT/'S7_Colab_PublicSource_RunStatus.json').write_text(json.dumps(STATUS,indent=2))
print('Public source inventory:',len(SOURCE_ROWS),'entries;',len(ERROR_ROWS),'metadata retrieval issues')
print('Author GitHub BED paths:',STATUS.get('author_repo_bed_paths','not verified (request failed)'))
print('Exact author HAR coordinates available in embedded files? NO')


In [ ]:

# OPTIONAL: if an actual original-author hg38 BED was obtained later, compare it.
# Default: this cell DOES NOT require uploads. To use a BED, set AUTHOR_BED_PATH to its local path.
# Author provenance must be reviewed: file upload / name alone does NOT make coordinates primary-confirmed.
AUTHOR_BED_PATH=None  # e.g. '/content/hars_hg38.bed' (original author file only; otherwise leave None)
if AUTHOR_BED_PATH is not None and Path(AUTHOR_BED_PATH).exists():
    records=defaultdict(list)
    for line in Path(AUTHOR_BED_PATH).read_text().splitlines():
        if not line or line.startswith('#') or line.startswith('track ') or line.startswith('browser '):continue
        p=line.split('\t')
        if len(p)<4: continue
        try: chrom,beg,end,name=p[0],int(p[1]),int(p[2]),p[3]
        except Exception: continue
        if beg>=end:continue
        records[name].append((chrom,beg,end))
    comparison=[]
    for r in cross:
        name=r['screen_HAR']; rr=records.get(name,[])
        if len(rr)!=1: st='MISSING_NAME_OR_NONUNIQUE_AUTHOR_BED'
        elif not r['human_hg38_chrom']:st='SOURCE_BED_FOUND_BUT_CATALOG_UNRESOLVED'
        else:
            chrom,beg,end=rr[0]
            s=int(r['human_start0']);e=int(r['human_end0_exclusive'])
            if chrom!=r['human_hg38_chrom']:st='CHROMOSOME_CONFLICT'
            elif beg==s and end==e:st='IDENTICAL_INTERVAL_PROVENANCE_REVIEW_PENDING'
            elif min(end,e)>max(beg,s):st='OVERLAPPING_DIFFERENT_BOUNDARIES_PROVENANCE_REVIEW_PENDING'
            else:st='NONOVERLAPPING_COORDINATE_CONFLICT'
        comparison.append({'screen_HAR':name,'primary_bed_row_count_for_exact_name':len(rr),
                           'candidate_primary_bed_position':str(rr[0]) if len(rr)==1 else '',
                           'provisional_catalog_position':str((r['human_hg38_chrom'],r['human_start0'],r['human_end0_exclusive'])),
                           'comparison_status':st,'verified_original_author_provenance':'NOT_VERIFIED_BY_UPLOADING'})
    savecsv('S8_Optional_PrimaryBED_NameAndCoordinate_Comparison.csv',comparison)
    print('Compared',len(comparison),'names to input BED. UPLOAD ALONE DOES NOT VERIFY AUTHOR PROVENANCE.')
else:
    print('Primary-author BED not provided. No identities promoted to confirmed.')


In [ ]:
# Create downloadable output ZIP without Google Drive
readme = """STAGE 19B - SOURCE INVENTORY AND DESIGN RULE
Original Noble ±300bp design used as a compatibility gate, not confirmation of original coordinates.
There are four beyond-300 guides in three loci (HACNS336, HACNS643, HAR169).
The other 28 HAR labels have no provisional catalog ortholog; they were not guide-distance tested.
Exact original author HAR BED remains missing until retrieved and its provenance verified.
"""
(OUT/'README_ResearchInterpretation.txt').write_text(readme)
zipout=ROOT/'HAR_Stage19B_Colab_Source_Discovery_Results.zip'
with zipfile.ZipFile(zipout,'w',zipfile.ZIP_DEFLATED,compresslevel=8) as z:
    for p in sorted(OUT.iterdir()):
        if p.is_file(): z.write(p,p.name)
print('Saved:',zipout,'size:',zipout.stat().st_size,'bytes')
try:
    from google.colab import files
    files.download(str(zipout))
except ImportError:
    print('Local run; result saved at',zipout)
